In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup, Comment
import re
import time

In [2]:
conversion_table = {
    "ATL": "Atlanta Hawks",
    "BOS": "Boston Celtics",
    "CHO": "Charlotte Hornets",
    "CHI": "Chicago Bulls",
    "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors",
    "HOU": "Houston Rockets",
    "IND": "Indiana Pacers",
    "LAC": "Los Angeles Clippers",
    "LAL": "Los Angeles Lakers",
    "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "NOP": "New Orleans Pelicans",
    "NYK": "New York Knicks",
    "BRK": "Brooklyn Nets",
    "OKC": "Oklahoma City Thunder",
    "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers",
    "PHO": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "SAC": "Sacramento Kings",
    "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz",
    "WAS": "Washington Wizards",
    "SAS": "San Antonio Spurs",
}


def normalize(data):
    total = sum(data)
    normalized_data = [x / total for x in data]
    return normalized_data

weights = [
    1.5,
    1.8,
    1.5,
    None,
    None,
    None,
    None,
]  # offdef_ratings, injuries, momentum, historical_performances, fatigue, momentum, sentiment, advanced

In [130]:
class Team:
    def __init__(self, team_abbr):
        self.offdef_ratings = self.find_offdef_ratings(team_abbr)
        self.team_abbr = team_abbr
        self.df = pd.read_html(
            "https://www.basketball-reference.com/teams/%s/2024.html" % team_abbr
        )
        self.sleep = time.sleep(4)  # sleep between two calls
        self.game_df = pd.read_html(
            "https://www.basketball-reference.com/teams/%s/2024_games.html" % team_abbr
        )
        self.injuries = self.find_injuries()
        self.player_scores = self.calculate_player_scores()
        # self.team_score_with, self.team_score_without = self.calculate_team_score()
        self.momentum = self.calculate_momentum()
        self.overall_with, self.overall_without = self.calculate_overall()

    def calculate_player_overall(self):
        pass

    def calculate_overall(self):
        """
        return (float) : team overall with and team overall without day to day players
        """
        # with
        overall_with = 0
        with_counter = 0
        for player in self.player_scores:
            if player not in self.injuries[0] and player not in self.injuries[1]:
                overall_with += self.player_scores[player]
                with_counter += 1
        overall_with = overall_with / with_counter

        # without

        overall_without = 0
        without_counter = 0
        for player in self.player_scores:
            if player not in self.injuries[0]:
                overall_without += self.player_scores[player]
                without_counter += 1
        overall_without = overall_without / without_counter

        # overall_with = overall_with + self.offdef_ratings + (self.momentum * 2)
        # overall_without = overall_without + self.offdef_ratings + (self.momentum * 2)

        print("Day to Day Players (check for most recent update): ", self.injuries[1])

        return overall_with, overall_without

    def calculate_player_scores(self):
        """
        return (dict): player --> sum of weights
        """
        player_weights = [
            0.2,
            0.2,
            0.2,
            0.2,
            0.2,
        ]  # points, assists, rebounds, win shares, value over replacement player

        player_scores = {}

        per_game_df = self.df[1].loc[
            self.df[1]["Player"].isin(self.df[3].head(13)["Player"])
        ]
        advanced_stats_df = (
            self.df[3]
            .head(13)
            .loc[self.df[3].head(13)["Player"].isin(self.df[1]["Player"])]
        )

        for index, player in per_game_df.iterrows():  # pts, ast, rebs
            name = player["Player"]
            player_scores[name] = []
            player_scores[name].append(player["PTS"])
            player_scores[name].append(player["AST"])
            player_scores[name].append(player["TRB"])

        for index, player in advanced_stats_df.iterrows():
            name = player["Player"]
            player_scores[name].append(player["BPM"])
            player_scores[name].append(player["VORP"])

        scores = []
        for player in player_scores:
            for i in range(len(player_scores[player])):
                player_scores[player][i] = player_scores[player][i] * player_weights[i]
            player_scores[player] = sum(player_scores[player])
            scores.append(player_scores[player])

        # scores = normalize(scores)

        for i, player in enumerate(player_scores):
            player_scores[player] = scores[i]

        return player_scores

    def calculate_importance(self, player):
        """
        player (pd.DataFrame) : dataframe of player on/off stats
        return ()
        """
        pass

    def calculate_team_score(self):
        """
        players_scores (dict): player: value
        injuries (list) : [[out players], [day to day players]]
        return (float): team score with, team score without
        """

        team_score_with = 1  # consider day to day players as out
        team_score_without = 1  # don't consider day to day players as out

        for lst in self.injuries:
            for player in lst:
                if (
                    player not in self.player_scores
                ):  # for whatever reason, player doesnt show up on roster/injury report
                    pass
                else:
                    score = self.player_scores[player]
                    team_score_with -= score

        for player in self.injuries[0]:
            if player not in self.player_scores:
                pass
            else:
                score = self.player_scores[player]
                team_score_without -= score
        if len(self.injuries[1]) == 0:
            return team_score_with, team_score_without
        else:
            print(
                "Day to Day Players (check for most recent update): ", self.injuries[1]
            )
            return team_score_with, team_score_without

    def find_injuries(self):
        """
        return (dict): team: [[out players], [day to day players]]
        """
        time.sleep(4)
        df = pd.read_html("https://www.basketball-reference.com/friv/injuries.fcgi")
        injuries = {}
        for index, player in df[0].iterrows():
            if player["Team"] not in injuries:
                if "Day To Day" in player["Description"].split("-")[0]:
                    injuries[player["Team"]] = [[], [player["Player"]]]
                else:
                    injuries[player["Team"]] = [[player["Player"]], []]
            else:
                if "Day To Day" in player["Description"].split("-")[0]:
                    injuries[player["Team"]][1].append(player["Player"])
                else:  # player is out
                    injuries[player["Team"]][0].append(player["Player"])
        if conversion_table[self.team_abbr] in injuries:
            return injuries[conversion_table[self.team_abbr]]
        else:
            return [[], []]

    def find_offdef_ratings(self, team_abbr):
        """
        team_abbr (str) : team abbreivation (CHI, BOS, LAL)
        return (list[float]) : [off_rtg, def_rtg]
        """
        constant = 200  # for inverse relationship for defensive ratings
        url = "https://www.basketball-reference.com/teams/%s/2024.html" % team_abbr
        r = requests.get(url)
        soup = BeautifulSoup(r.text, "html.parser")
        tag = soup.find("div", {"id": "all_team_misc"})

        for element in tag(text=lambda text: isinstance(text, Comment)):
            soup = BeautifulSoup(element, "html.parser")

        rtgs = soup.find_all("td", {"data-stat": ["off_rtg", "def_rtg"]})
        rtgs = (float(rtgs[0].get_text()) + constant - float(rtgs[1].get_text())) / 3
        # rtgs = np.log2(
        #     [float(rtgs[0].get_text()), constant - float(rtgs[1].get_text())]
        # )
        time.sleep(4)
        print(rtgs)
        return rtgs

    def calculate_momentum(self):
        """
        return (float) : win percentage of last x number of games
        """
        df = self.game_df[0]
        df = df[df["Unnamed: 7"].notna()]
        games_prior = 7  # number of games prior
        last_games = df["Unnamed: 7"].tail(games_prior).tolist()
        games_won = 0
        for game_result in last_games:
            if game_result == "W":
                games_won += 1
        win_percentage = games_won / games_prior

        return win_percentage

    def pipeline(self):
        """
        return (list): [offdef_ratings, injuries, historical_performances, fatigue, homecourt_advantage]
        """
        data = [
            sum(self.offdef_ratings) * weights[0],
            self.team_abbr,
            self.team_score_with * weights[1],
            self.team_score_without * weights[1],
            self.momentum * weights[2],
        ]

        return data

In [116]:
class Game:
    def __init__(self, home_team, away_team):
        self.home_team = home_team
        self.away_team = away_team

    def predict(self):
        """
        include_day_to_day (boolean) : decide whether to consider day to day players as out
        """
        home_team_sum_with = (
            sum(self.home_team.offdef_ratings) * weights[0]
            + self.home_team.team_score_with * weights[1]
            + 0.3
            + self.home_team.momentum * weights[2]
        )
        away_team_sum_with = (
            sum(self.away_team.offdef_ratings) * weights[0]
            + self.away_team.team_score_with * weights[1]
            + self.away_team.momentum * weights[2]
        )
        home_team_sum_without = (
            sum(self.home_team.offdef_ratings) * weights[0]
            + self.home_team.team_score_without * weights[1]
            + 0.3
            + self.home_team.momentum * weights[2]
        )
        away_team_sum_without = (
            sum(self.away_team.offdef_ratings) * weights[0]
            + self.away_team.team_score_without * weights[1]
            + self.away_team.momentum * weights[2]
        )

        if home_team_sum_with > away_team_sum_with:
            print("Winner with Day to Day Players out: " + self.home_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_with)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_with)
            )
        else:
            print("Winner with Day to Day Players out: " + self.away_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_with)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_with)
            )

        print("")

        if home_team_sum_without > away_team_sum_without:
            print("Winner with Day to Day Players out: " + self.home_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_without)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_without)
            )
        else:
            print("Winner with Day to Day Players out: " + self.away_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_without)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_without)
            )

### Outcome Testing

In [131]:
sixers = Team("PHI")
print(sixers.overall_with)
print(sixers.overall_without)

69.6
Day to Day Players (check for most recent update):  []
3.6738461538461538
3.6738461538461538


In [110]:
bulls = Team("CHI")
print(bulls.overall_with)
print(bulls.overall_without)

65.23333333333333
Day to Day Players (check for most recent update):  ['Alex Caruso']
68.62164502164502
68.76119047619048


In [118]:
timberwolves = Team("MIN")
print(timberwolves.overall_with)
print(timberwolves.overall_without)

68.96666666666667
Day to Day Players (check for most recent update):  ['Anthony Edwards']
73.58928571428571
73.96710622710623


In [71]:
bulls = Team("CHI")

[6.81121412 6.38197548]
{'DeMar DeRozan': [22.3, 5.3, 3.5], 'Zach LaVine': [21.0, 3.4, 4.9], 'Nikola Vučević': [16.4, 3.3, 10.5], 'Coby White': [16.7, 4.5, 3.6], 'Patrick Williams': [8.8, 1.3, 4.2], 'Alex Caruso': [9.5, 2.3, 3.4], 'Torrey Craig': [5.8, 1.1, 5.0], 'Ayo Dosunmu': [6.3, 1.7, 1.7], 'Jevon Carter': [5.4, 1.2, 0.7], 'Andre Drummond': [6.1, 0.6, 6.9], 'Dalen Terry': [1.3, 0.7, 1.1], 'Julian Phillips': [0.8, 0.4, 0.7], 'Terry Taylor': [0.7, 0.0, 0.3]}
Day to Day Players (check for most recent update):  ['Alex Caruso']


In [73]:
print(bulls.overall_with)
print(bulls.overall_without)

2.2454545454545456
2.3850000000000002


In [128]:
np.emath.logn(5, 110.3 + 200- 116.6)

3.272142733638123

In [129]:
np.emath.logn(5, 120.3 + 200 - 110.6)

3.3214564517267497

In [123]:
s = 110.3 + 200 - 116.6
s / 15

12.913333333333334

In [124]:
s = 120.3 + 200 - 110.6
s / 10

20.970000000000002

In [57]:
pistons = Team("DET")

[6.76022095 6.34872815]
{'Cade Cunningham': [22.0, 7.3, 3.9], 'Isaiah Stewart': [10.8, 1.4, 7.1], 'Jalen Duren': [12.6, 2.5, 10.9], 'Ausar Thompson': [10.6, 2.5, 8.4], 'Killian Hayes': [9.8, 4.4, 3.0], 'Jaden Ivey': [11.5, 2.7, 2.7], 'Isaiah Livers': [6.1, 1.3, 2.0], 'Alec Burks': [10.7, 1.7, 2.3], 'Marvin Bagley III': [9.8, 1.2, 4.8], 'Marcus Sasser': [7.7, 2.7, 1.8], 'Kevin Knox': [6.6, 0.4, 2.8], 'James Wiseman': [4.9, 0.4, 3.2], 'Stanley Umude': [5.0, 0.5, 1.9]}


In [70]:
bulls.injuries

[['Lonzo Ball', 'Zach LaVine'], ['Alex Caruso']]

In [58]:
bucks = Team("MIL")
bulls = Team("CHI")

[6.9068906  6.38197548]
{'Damian Lillard': [25.0, 7.0, 4.6], 'Giannis Antetokounmpo': [30.6, 5.1, 10.7], 'Brook Lopez': [13.7, 1.5, 5.0], 'Malik Beasley': [12.2, 1.4, 4.3], 'Jae Crowder': [8.1, 1.7, 3.9], 'Bobby Portis': [11.3, 1.1, 6.6], 'Pat Connaughton': [5.8, 2.0, 3.5], 'Khris Middleton': [12.4, 4.2, 4.8], 'Cameron Payne': [6.6, 2.4, 1.3], 'MarJon Beauchamp': [5.2, 0.9, 2.4], 'A.J. Green': [3.5, 0.8, 0.9], 'Andre Jackson Jr.': [2.0, 0.6, 1.2], 'Thanasis Antetokounmpo': [1.0, 0.5, 0.6]}
[6.81121412 6.38197548]
{'DeMar DeRozan': [22.3, 5.3, 3.5], 'Zach LaVine': [21.0, 3.4, 4.9], 'Nikola Vučević': [16.4, 3.3, 10.5], 'Coby White': [16.7, 4.5, 3.6], 'Patrick Williams': [8.8, 1.3, 4.2], 'Alex Caruso': [9.5, 2.3, 3.4], 'Torrey Craig': [5.8, 1.1, 5.0], 'Ayo Dosunmu': [6.3, 1.7, 1.7], 'Jevon Carter': [5.4, 1.2, 0.7], 'Andre Drummond': [6.1, 0.6, 6.9], 'Dalen Terry': [1.3, 0.7, 1.1], 'Julian Phillips': [0.8, 0.4, 0.7], 'Terry Taylor': [0.7, 0.0, 0.3]}
Day to Day Players (check for most recen

In [60]:
bucks.overall

3.226153846153847

In [61]:
bulls.overall

2.6584615384615384

In [ ]:
sixers.overall

2.5855555555555556

In [ ]:
bucks.player_scores

{'Damian Lillard': 7.920000000000001,
 'Giannis Antetokounmpo': 11.000000000000002,
 'Brook Lopez': 4.62,
 'Malik Beasley': 3.6399999999999997,
 'Jae Crowder': 3.0000000000000004,
 'Bobby Portis': 3.2200000000000006,
 'Pat Connaughton': 1.7800000000000002,
 'Khris Middleton': 4.32,
 'Cameron Payne': 1.46,
 'MarJon Beauchamp': 1.0599999999999998,
 'A.J. Green': 0.9,
 'Andre Jackson Jr.': 0.28,
 'Thanasis Antetokounmpo': -1.2600000000000002,
 'Chris Livingston': 1.6800000000000002,
 'Robin Lopez': -2.12,
 'Marques Bolden': -0.7000000000000001,
 'TyTy Washington Jr.': -1.7400000000000002,
 'Lindell Wigginton': -11.8}

In [ ]:
sixers.player_scores

{'Tyrese Maxey': 8.940000000000001,
 'Tobias Harris': 5.06,
 'Joel Embiid': 12.959999999999999,
 "De'Anthony Melton": 4.319999999999999,
 'Kelly Oubre Jr.': 4.2,
 'Nicolas Batum': 3.04,
 'P.J. Tucker': 0.5800000000000001,
 'Patrick Beverley': 1.9400000000000002,
 'Robert Covington': 2.940000000000001,
 'Paul Reed': 1.98,
 'Danuel House Jr.': 0.7400000000000001,
 'Marcus Morris': 1.5,
 'Jaden Springer': 0.06000000000000029,
 'Danny Green': -0.9800000000000002,
 'Furkan Korkmaz': 0.08000000000000007,
 'Mo Bamba': 1.12,
 'KJ Martin': -1.14,
 'Filip Petrušev': -0.8}

In [ ]:
def calculate_player_overall():
    

In [ ]:
sixers = Team("PHI")

[6.93310047 6.43629512]


In [ ]:
sixers.player_scores

{'Tyrese Maxey': 0.1920928233777396,
 'Tobias Harris': 0.10872367855608078,
 'Joel Embiid': 0.27847013321873654,
 "De'Anthony Melton": 0.09282337773957884,
 'Kelly Oubre Jr.': 0.09024495058014612,
 'Nicolas Batum': 0.06532015470562957,
 'P.J. Tucker': 0.012462397937258275,
 'Patrick Beverley': 0.0416845724108294,
 'Robert Covington': 0.0631714654061023,
 'Paul Reed': 0.04254404813064031,
 'Danuel House Jr.': 0.015900300816501935,
 'Marcus Morris': 0.03223033949290933,
 'Jaden Springer': 0.0012892135797163793,
 'Danny Green': -0.02105715513536743,
 'Furkan Korkmaz': 0.0017189514396218322,
 'Mo Bamba': 0.02406532015470563,
 'KJ Martin': -0.024495058014611087,
 'Filip Petrušev': -0.017189514396218308}

In [ ]:
sixers.injuries

[[], []]

In [ ]:
sixers = Team("PHI")
pistons = Team("DET")
Game(pistons, sixers).predict()

[6.93310047 6.43629512]
[6.76022095 6.34872815]
Day to Day Players (check for most recent update):  ['Marvin Bagley III', 'Jalen Duren']
Winner with Day to Day Players out: PHI
DET: 21.390366656227727 vs PHI: 22.925521963584668

Winner with Day to Day Players out: PHI
DET: 21.76342365104638 vs PHI: 22.925521963584668


In [ ]:
sixers.pipeline()

[13.36939559477073, 'PHI', 1, 1, 0.7142857142857143]

In [ ]:
hawks = Team("ATL")
raptors = Team("TOR")
Game(raptors, hawks).predict()

[6.90086681 6.32192809]
Day to Day Players (check for most recent update):  ["De'Andre Hunter"]
[6.81378119 6.41953889]
Day to Day Players (check for most recent update):  ['Chris Boucher', 'Otto Porter Jr.']
Winner with Day to Day Players out: TOR
TOR: 22.090932505048617 vs ATL: 21.78596920962261

Winner with Day to Day Players out: TOR
TOR: 22.16426583838195 vs ATL: 22.01176651649717


In [ ]:
hornets = Team("CHO")
heat = Team("MIA")
Game(heat, hornets).predict()

Day to Day Players (check for most recent update):  ['Mark Williams']
Winner with Day to Day Players out: MIA
MIA: 5.168097497360562 vs CHO: 4.893118591057709

Winner with Day to Day Players out: MIA
MIA: 5.168097497360562 vs CHO: 4.927700264364482


In [ ]:
lakers = Team("LAL")
spurs = Team("SAS")
Game(spurs, lakers).predict()

Winner with Day to Day Players out: SAS
SAS: 5.108990846827633 vs LAL: 5.0835346673352255

Winner with Day to Day Players out: SAS
SAS: 5.108990846827633 vs LAL: 5.0835346673352255


In [ ]:
pacers = Team("IND")
bucks = Team("MIL")
Game(bucks, pacers).predict()

Winner with Day to Day Players out: MIL
MIL: 5.225303943482405 vs IND: 5.054189173658766

Winner with Day to Day Players out: MIL
MIL: 5.225303943482405 vs IND: 5.054189173658766


In [ ]:
grizzlies = Team("MEM")
rockets = Team("HOU")
Game(rockets, grizzlies).predict()

Day to Day Players (check for most recent update):  ['Reggie Bullock', 'Tari Eason', 'Amen Thompson']
Winner with Day to Day Players out: HOU
HOU: 5.264232626923988 vs MEM: 4.969154616212659

Winner with Day to Day Players out: HOU
HOU: 5.300886936469864 vs MEM: 4.969154616212659


In [ ]:
knicks = Team("NYK")
jazz = Team("UTA")
Game(jazz, knicks).predict()

Day to Day Players (check for most recent update):  ['Immanuel Quickley']
Day to Day Players (check for most recent update):  ['John Collins', 'Walker Kessler', 'Lauri Markkanen']
Winner with Day to Day Players out: UTA
UTA: 5.018825496936471 vs NYK: 4.985152860732536

Winner with Day to Day Players out: UTA
UTA: 5.184556896684264 vs NYK: 5.0448387246068815


In [ ]:
nets = Team("BRK")
suns = Team("PHO")
Game(suns, nets).predict()

Winner with Day to Day Players out: PHO
PHO: 5.153225694535496 vs BRK: 5.0188977202917044

Winner with Day to Day Players out: PHO
PHO: 5.153225694535496 vs BRK: 5.0188977202917044


### Code Testing

In [49]:
df = pd.read_html("https://www.basketball-reference.com/teams/PHI/2024.html")

In [51]:
df[3].head(13).loc[df[3].head(13)['Player'].isin(df[1]['Player'])]

,Rk,Player,Age,G,MP,PER,TS%,3PAr,FTr,ORB%,...,Unnamed: 17,OWS,DWS,WS,WS/48,Unnamed: 22,OBPM,DBPM,BPM,VORP
0,1,Tyrese Maxey,23,21,793,22.3,0.606,0.414,0.284,1.7,...,NaN,2.9,0.6,3.5,0.212,NaN,6.2,-0.7,5.4,1.5
1,2,Tobias Harris,31,22,767,14.8,0.599,0.240,0.287,3.1,...,NaN,1.0,0.7,1.8,0.111,NaN,0.1,-0.7,-0.6,0.3
2,3,De'Anthony Melton,25,22,663,13.5,0.545,0.575,0.258,2.9,...,NaN,0.7,0.8,1.5,0.107,NaN,0.0,0.6,0.5,0.4
3,4,Joel Embiid,29,19,656,33.1,0.633,0.158,0.545,10.5,...,NaN,2.9,1.1,4.0,0.292,NaN,8.5,2.8,11.3,2.2
4,5,Patrick Beverley,35,22,395,11.1,0.546,0.527,0.198,5.6,...,NaN,0.5,0.4,0.8,0.103,NaN,-1.2,0.3,-0.9,0.1
5,6,Nicolas Batum,35,13,335,13.0,0.778,0.774,0.208,5.0,...,NaN,0.8,0.3,1.1,0.157,NaN,1.4,0.5,1.9,0.3
6,7,Paul Reed,24,22,308,15.6,0.625,0.119,0.107,13.8,...,NaN,0.5,0.4,0.9,0.140,NaN,-1.0,0.5,-0.5,0.1
7,8,Robert Covington,33,18,305,16.5,0.636,0.577,0.296,9.1,...,NaN,0.6,0.5,1.2,0.186,NaN,1.5,2.8,4.2,0.5
8,9,Kelly Oubre Jr.,28,11,293,16.9,0.599,0.403,0.277,5.7,...,NaN,0.5,0.3,0.9,0.144,NaN,1.1,-0.2,0.9,0.2
9,10,Marcus Morris,34,15,188,13.0,0.615,0.452,0.081,3.6,...,NaN,0.3,0.2,0.5,0.123,NaN,-0.4,-0.2,-0.6,0.1


In [52]:
df[1].loc[df[1]["Player"].isin(df[3].head(13)["Player"])]

,Rk,Player,Age,G,GS,MP,FG,FGA,FG%,3P,...,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1,Tyrese Maxey,23,21,21,37.8,9.3,19.8,0.470,3.3,...,0.898,0.6,3.6,4.1,6.7,0.8,0.6,1.4,1.9,27.0
1,2,Tobias Harris,31,22,22,34.9,6.4,12.5,0.509,1.0,...,0.886,1.0,5.1,6.0,2.7,1.0,0.3,1.8,1.7,16.9
2,3,Joel Embiid,29,19,19,34.5,11.1,21.3,0.520,1.1,...,0.877,3.3,8.2,11.5,6.4,1.1,1.9,3.9,2.9,33.4
3,4,De'Anthony Melton,25,22,22,30.1,4.2,10.6,0.395,2.3,...,0.800,0.8,3.3,4.0,3.8,1.5,0.5,1.4,2.5,12.9
4,5,Kelly Oubre Jr.,28,11,5,26.6,5.4,10.8,0.496,1.5,...,0.758,1.4,3.4,4.7,0.7,1.3,0.4,0.7,2.2,14.5
5,6,Nicolas Batum,35,13,10,25.8,2.4,4.1,0.585,1.6,...,0.636,1.2,2.9,4.1,2.0,0.5,0.7,0.5,1.9,6.9
7,8,Patrick Beverley,35,22,1,18.0,1.8,4.1,0.440,0.7,...,0.722,0.9,2.2,3.1,2.5,0.5,0.3,0.7,1.9,4.9
8,9,Robert Covington,33,18,3,16.9,1.8,3.9,0.465,0.9,...,0.905,1.4,2.3,3.7,0.6,1.3,0.7,0.4,1.8,5.7
9,10,Paul Reed,24,22,1,14.0,2.3,3.8,0.595,0.2,...,0.667,1.7,2.5,4.3,1.0,0.5,0.6,0.9,2.1,5.0
10,11,Danuel House Jr.,30,11,0,12.5,1.5,3.2,0.457,0.6,...,0.750,0.4,1.0,1.4,0.8,0.5,0.1,0.8,0.9,4.4


In [33]:
df[1].head(11)

,Rk,Player,Age,G,GS,MP,FG,FGA,FG%,3P,...,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1,Tyrese Maxey,23,21,21,37.8,9.3,19.8,0.470,3.3,...,0.898,0.6,3.6,4.1,6.7,0.8,0.6,1.4,1.9,27.0
1,2,Tobias Harris,31,22,22,34.9,6.4,12.5,0.509,1.0,...,0.886,1.0,5.1,6.0,2.7,1.0,0.3,1.8,1.7,16.9
2,3,Joel Embiid,29,19,19,34.5,11.1,21.3,0.520,1.1,...,0.877,3.3,8.2,11.5,6.4,1.1,1.9,3.9,2.9,33.4
3,4,De'Anthony Melton,25,22,22,30.1,4.2,10.6,0.395,2.3,...,0.800,0.8,3.3,4.0,3.8,1.5,0.5,1.4,2.5,12.9
4,5,Kelly Oubre Jr.,28,11,5,26.6,5.4,10.8,0.496,1.5,...,0.758,1.4,3.4,4.7,0.7,1.3,0.4,0.7,2.2,14.5
5,6,Nicolas Batum,35,13,10,25.8,2.4,4.1,0.585,1.6,...,0.636,1.2,2.9,4.1,2.0,0.5,0.7,0.5,1.9,6.9
6,7,P.J. Tucker,38,3,3,22.0,0.7,1.7,0.400,0.7,...,NaN,0.3,4.3,4.7,0.0,1.0,0.7,0.7,3.0,2.0
7,8,Patrick Beverley,35,22,1,18.0,1.8,4.1,0.440,0.7,...,0.722,0.9,2.2,3.1,2.5,0.5,0.3,0.7,1.9,4.9
8,9,Robert Covington,33,18,3,16.9,1.8,3.9,0.465,0.9,...,0.905,1.4,2.3,3.7,0.6,1.3,0.7,0.4,1.8,5.7
9,10,Paul Reed,24,22,1,14.0,2.3,3.8,0.595,0.2,...,0.667,1.7,2.5,4.3,1.0,0.5,0.6,0.9,2.1,5.0


In [34]:
df[3].head(11)

,Rk,Player,Age,G,MP,PER,TS%,3PAr,FTr,ORB%,...,Unnamed: 17,OWS,DWS,WS,WS/48,Unnamed: 22,OBPM,DBPM,BPM,VORP
0,1,Tyrese Maxey,23,21,793,22.3,0.606,0.414,0.284,1.7,...,NaN,2.9,0.6,3.5,0.212,NaN,6.2,-0.7,5.4,1.5
1,2,Tobias Harris,31,22,767,14.8,0.599,0.240,0.287,3.1,...,NaN,1.0,0.7,1.8,0.111,NaN,0.1,-0.7,-0.6,0.3
2,3,De'Anthony Melton,25,22,663,13.5,0.545,0.575,0.258,2.9,...,NaN,0.7,0.8,1.5,0.107,NaN,0.0,0.6,0.5,0.4
3,4,Joel Embiid,29,19,656,33.1,0.633,0.158,0.545,10.5,...,NaN,2.9,1.1,4.0,0.292,NaN,8.5,2.8,11.3,2.2
4,5,Patrick Beverley,35,22,395,11.1,0.546,0.527,0.198,5.6,...,NaN,0.5,0.4,0.8,0.103,NaN,-1.2,0.3,-0.9,0.1
5,6,Nicolas Batum,35,13,335,13.0,0.778,0.774,0.208,5.0,...,NaN,0.8,0.3,1.1,0.157,NaN,1.4,0.5,1.9,0.3
6,7,Paul Reed,24,22,308,15.6,0.625,0.119,0.107,13.8,...,NaN,0.5,0.4,0.9,0.140,NaN,-1.0,0.5,-0.5,0.1
7,8,Robert Covington,33,18,305,16.5,0.636,0.577,0.296,9.1,...,NaN,0.6,0.5,1.2,0.186,NaN,1.5,2.8,4.2,0.5
8,9,Kelly Oubre Jr.,28,11,293,16.9,0.599,0.403,0.277,5.7,...,NaN,0.5,0.3,0.9,0.144,NaN,1.1,-0.2,0.9,0.2
9,10,Marcus Morris,34,15,188,13.0,0.615,0.452,0.081,3.6,...,NaN,0.3,0.2,0.5,0.123,NaN,-0.4,-0.2,-0.6,0.1


In [96]:
df

,G,Date,Start (ET),Unnamed: 3,Unnamed: 4,Unnamed: 5,Opponent,Unnamed: 7,Unnamed: 8,Tm,Opp,W,L,Streak,Notes
0,1,"Wed, Oct 25, 2023",9:00p,NaN,Box Score,@,Utah Jazz,W,NaN,130,114,1,0,W 1,NaN
1,2,"Fri, Oct 27, 2023",10:00p,NaN,Box Score,NaN,Golden State Warriors,L,NaN,114,122,1,1,L 1,NaN
2,3,"Sun, Oct 29, 2023",9:00p,NaN,Box Score,NaN,Los Angeles Lakers,W,OT,132,127,2,1,W 1,NaN
3,4,"Wed, Nov 1, 2023",10:00p,NaN,Box Score,@,Golden State Warriors,L,NaN,101,102,2,2,L 1,NaN
4,5,"Sat, Nov 4, 2023",8:00p,NaN,Box Score,@,Houston Rockets,L,NaN,89,107,2,3,L 2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,79,"Tue, Apr 9, 2024",8:00p,NaN,NaN,@,Oklahoma City Thunder,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,80,"Thu, Apr 11, 2024",10:00p,NaN,NaN,NaN,New Orleans Pelicans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,G,Date,Start (ET),NaN,NaN,NaN,Opponent,NaN,NaN,Tm,Opp,W,L,Streak,Notes
84,81,"Fri, Apr 12, 2024",10:30p,NaN,NaN,NaN,Phoenix Suns,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
df = df[df["Unnamed: 7"].notna()]

In [102]:
df["Unnamed: 7"].tail(7).tolist()

['W', 'W', 'L', 'W', 'L', 'W', 'W']

In [ ]:
df = pd.read_html("https://www.basketball-reference.com/friv/injuries.fcgi")

## Find players that are out (cbssports method)

In [ ]:
# def find_team_name(tag):
#     pattern = r'>(.*?)<\/a>'
#     team_name = str(tag.find_all(class_ = "TeamName")[0])
#     team_name = re.search(pattern, team_name).group(1)
#     team_name = team_name.split('>', 1)[-1]
#     return team_name

In [ ]:
# def find_injury_time(tag):
#     element = str(tag).split("width: 40%;")[1]
#     start_index = element.find('">') + 2
#     end_index = element.find('</td>')
#     injury_time = element[start_index:end_index].strip()
#     if injury_time != "Game Time Decision":
#         return "Out"
#     else:
#         return "Questionable"

In [ ]:
# def find_injuries(teams_element):
#     injuries = {}
#     for team_element in teams_element:
#         team_name = find_team_name(team_element)
#         injuries[team_name] = []
#         for report_element in team_element.find_all('tr', class_='TableBase-bodyTr'):
#             player_name = report_element.find('span', class_='CellPlayerName--long').text
#             # injury_time = find_injury_time(report_element)
#             injuries[team_name].append(player_name)
#     return injuries


In [ ]:
# r = requests.get('https://www.cbssports.com/nba/injuries/')
# soup = BeautifulSoup(r.text, 'html.parser')
# teams = soup.find_all(class_ = 'TableBaseWrapper')

# injuries = find_injuries(teams)